# 00. Olist 데이터 마트 & 공통 전처리

Olist 원본 CSV를 로드한 뒤, **전체 분석 주제에서 공통으로 쓰는 전처리**를 적용하고  
`data/` 폴더에 저장합니다. **03~09번 분석 노트북은 이 노트북 실행 후 생성된 데이터를 사용합니다.**

## 1. 로드 및 경로 설정

In [ ]:
import os
import pandas as pd
from pathlib import Path

try:
    import kagglehub
    path = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
except Exception:
    path = Path(os.environ.get("OLIST_PATH", r"C:\Users\itwill\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2"))

DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(exist_ok=True)
print("원본 경로:", path)
print("저장 경로:", DATA_DIR)

In [ ]:
customers = pd.read_csv(path / "olist_customers_dataset.csv")
geolocation = pd.read_csv(path / "olist_geolocation_dataset.csv")
orders = pd.read_csv(path / "olist_orders_dataset.csv")
order_items = pd.read_csv(path / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(path / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(path / "olist_order_reviews_dataset.csv")
products = pd.read_csv(path / "olist_products_dataset.csv")
sellers = pd.read_csv(path / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(path / "product_category_name_translation.csv", encoding="utf-8-sig")

print("orders:", orders.shape, "| products:", products.shape)

## 2. 공통 전처리

- **orders**: 날짜 컬럼 `to_datetime`, delivered만 유지(선택용 테이블 별도 저장)
- **products**: 컬럼명 오타 수정, 카테고리 결측 'unknown'
- **order_payments**: not_defined → unknown
- **geolocation**: 우편번호당 1행(첫 행) 축소

In [ ]:
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in date_cols:
    if c in orders.columns:
        orders[c] = pd.to_datetime(orders[c], errors="coerce")

products = products.rename(columns={"product_name_lenght": "product_name_length", "product_description_lenght": "product_description_length"})
products["product_category_name"] = products["product_category_name"].fillna("unknown")

order_payments["payment_type"] = order_payments["payment_type"].replace("not_defined", "unknown")

geolocation_dedup = geolocation.drop_duplicates(subset=["geolocation_zip_code_prefix"], keep="first")

orders_delivered = orders[orders["order_status"] == "delivered"].copy()
print("orders_delivered:", len(orders_delivered), "| geolocation_dedup:", len(geolocation_dedup))

## 3. data/ 폴더에 저장

In [ ]:
customers.to_csv(DATA_DIR / "customers.csv", index=False, encoding="utf-8-sig")
geolocation_dedup.to_csv(DATA_DIR / "geolocation.csv", index=False, encoding="utf-8-sig")
orders.to_csv(DATA_DIR / "orders.csv", index=False, encoding="utf-8-sig")
orders_delivered.to_csv(DATA_DIR / "orders_delivered.csv", index=False, encoding="utf-8-sig")
order_items.to_csv(DATA_DIR / "order_items.csv", index=False, encoding="utf-8-sig")
order_payments.to_csv(DATA_DIR / "order_payments.csv", index=False, encoding="utf-8-sig")
order_reviews.to_csv(DATA_DIR / "order_reviews.csv", index=False, encoding="utf-8-sig")
products.to_csv(DATA_DIR / "products.csv", index=False, encoding="utf-8-sig")
sellers.to_csv(DATA_DIR / "sellers.csv", index=False, encoding="utf-8-sig")
category_translation.to_csv(DATA_DIR / "category_translation.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", list(DATA_DIR.glob("*.csv")))